<a href="https://colab.research.google.com/github/Abhi712-ui/lis501-ipynb-files/blob/main/LIS501_Week_3_Text_Representation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# LIS 501 · Week 3: Text Representation


**From last week to this week:** start with a text string, inspect its tokens and linguistic annotations, then represent each document using counts and weights.

Run and discuss the supplied code. In the **two in-class exercises**, edit the marked lines and explain what the output tells you. You do not need to memorize the syntax.

| Topic | What we will do |
|:--|:--|
| Regular expressions | Find character patterns; inspect the effects of replacements. |
| spaCy | Inspect tokens, lemmas, POS tags, and entities; **Exercise 1: count noun lemmas** (6 minutes). |
| Bag-of-Words | Build a document–term matrix; **Exercise 2: predict and check counts** (5 minutes). |
| TF-IDF | Calculate the lecture weights, compare library defaults, and interpret the result. |


## Before class: prepare Colab

Open this `.ipynb` file in Google Colab and save your own copy. Use a standard CPU runtime; no GPU, paid API, dataset upload, or Drive connection is needed for these examples. Run the four setup cells below before the demonstration. The package/model downloads require internet access and can take several minutes.

Use **Shift + Enter** to run a cell and move to the next one. Run cells from top to bottom. To write an interpretation, double-click the corresponding text cell. A new or restarted runtime forgets your Python variables, so rerun the setup/import cells and the preceding examples.

**After installation:** if Colab asks you to restart the session, do so, then run the imports and model check below. An import error after a package change is also a reason to restart. You do not need to understand or edit the installation commands.

The setup uses spaCy's 3.8 series and its compatible small English model. [Setup reference: spaCy models and languages](https://spacy.io/usage/models).

In [ ]:
# Setup 1 of 4: install the libraries. Run before the timed demo.
%pip -q install "spacy>=3.8,<3.9" scikit-learn pandas

In [ ]:
# Setup 2 of 4: download a compatible trained English model when needed.
# This is separate from installing the spaCy library.
import subprocess
import sys
from importlib import metadata

try:
    compatible_model = metadata.version("en_core_web_sm").startswith("3.8.")
except metadata.PackageNotFoundError:
    compatible_model = False

if compatible_model:
    print("A compatible English model is already installed.")
else:
    subprocess.run(
        [sys.executable, "-m", "spacy", "download", "en_core_web_sm"],
        check=True,
    )

In [ ]:
# Setup 3 of 4: tools used throughout the notebook.
import re
import sys
from collections import Counter
from math import log10

import numpy as np
import pandas as pd
import sklearn
from IPython.display import display
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer

print("Python:", sys.version.split()[0])
print("pandas:", pd.__version__)
print("scikit-learn:", sklearn.__version__)

In [ ]:
# Setup 4 of 4: load the trained pipeline and check its annotations.
import spacy
from spacy import displacy

try:
    nlp = spacy.load("en_core_web_sm")
except OSError as error:
    raise RuntimeError(
        "The English model could not be loaded. Run Setup 2, restart the "
        "Colab session if needed, and rerun Setups 3 and 4."
    ) from error

probe = nlp("Two friends visited a library.")
if not (probe.has_annotation("POS") and probe.has_annotation("LEMMA")
        and nlp.has_pipe("ner")):
    raise RuntimeError("Use the full en_core_web_sm pipeline for this demo.")

print("spaCy:", spacy.__version__)
print("English model:", nlp.meta["version"])
print("Pipeline components:", nlp.pipe_names)
print("Ready for the demo.")

## **1. Regular expressions**

**A pattern describes the characters we want to find.** Python's `re.findall(pattern, text)` returns the matching pieces. `re.sub(pattern, replacement, text)` returns a new string with matching pieces replaced.

We will keep the original string and save any changed version separately. This extends last week's work with strings, `.lower()`, and `.split()`.

[Reference: Python's `re` module](https://docs.python.org/3/library/re.html).

In [ ]:
# The welcome-string example from the original notebook.
text = "Welcome to LIS 501!"

print("Lowercase copy:", text.lower())
print("Capital-letter runs:", re.findall(r"[A-Z]+", text))
print("Digit runs:", re.findall(r"\d+", text))
print("Original:", text)

### Read the pattern in small pieces

| Pattern | Read it as |
|:--|:--|
| `[A-Z]` | One uppercase letter in the range A through Z. |
| `\d` | One digit. |
| `+` | One or more of the preceding item. |
| `\.` | A literal period. |
| `\s` | A whitespace character, including a space, tab, or newline. |

The prefix `r` in `r"\d+"` makes a **raw Python string**, so the backslash is passed through to the regex engine.

**Predict:** will `\d+` keep `15.2` together?

In [ ]:
# Revisit the O'Neal sentence from Week 2
stats_text = "O'Neal averaged 15.2 points, 9.2 rebounds and 1.0 assists per game."

print("Digits only:", re.findall(r"\d+", stats_text))
print("Decimal values:", re.findall(r"\d+\.\d+", stats_text))

The second pattern reads **digits → period → digits**. It matches the three decimal values in this sentence. It does not match a standalone whole number such as `9`; that would require a different rule. Its output is a list of **strings**, even when the strings look like numbers.

### Replace whitespace without deleting the original

The next example adapts the whitespace-cleaning example by adding a newline. `repr()` makes that newline visible as `\n`.

In [ ]:
raw_spacing = "Hello,   world!!!\nWelcome to regex."
clean_spacing = re.sub(r"\s+", " ", raw_spacing).strip()

print("Before:", repr(raw_spacing))
print("After: ", repr(clean_spacing))

### Inspect a cleaning rule before trusting it
`r"[^\w\s]"`: match characters that are **neither word characters nor whitespace**. Word characters include letters, digits, and underscores.

**Predict:** what happens when we apply that rule to the sports statistics? Does removing punctuation preserve the numeric values?

In [ ]:
# Deliberately inspect an unsuitable cleaning rule for this sentence.
stripped_stats = re.sub(r"[^\w\s]", "", stats_text)

print("Original:", stats_text)
print("Changed: ", stripped_stats)

**Notice:** `15.2` becomes `152`, and the apostrophe in `O'Neal` disappears. A short rule can make a consequential change. Likewise, replacing all whitespace removes line breaks that might matter in a poem or transcript. Choose transformations in relation to the text and the research question.

## Let's work with an example!

In [ ]:
email_demo = "It was a crisp fall morning at the University of Wisconsin-Madison, and the campus was buzzing with excitement as students and faculty prepared for the annual Data Science Symposium. In the bustling atrium of the Discovery Building, a small group of students gathered around a table piled high with pamphlets, coffee cups, and name tags. Among them was Alex Carter, a senior majoring in Computer Science. Alex had been working tirelessly on a project that would be showcased at the event, and he was eager to share his work with anyone who would listen. As he sipped his coffee, Alex received an email notification on his phone from his project advisor, Dr. Sarah Thompson. The message read: \"Alex, please send the latest draft of your presentation slides to sarah.thompson@wisc.edu by noon. Looking forward to seeing your progress!\" Alex quickly typed a reply, making sure to attach the slides. He glanced around the room, spotting his friend and fellow data enthusiast, Priya Patel, who was intently discussing her research on machine learning models with a group of underclassmen. Priya’s research was focused on improving predictive algorithms, and she was always willing to lend a helping hand to those interested in the field. Alex made a mental note to catch up with her later and send her an email at priya.patel@wisc.edu to get her feedback on his presentation. Not far from them, Michael Rodriguez was deeply engrossed in a conversation with Dr. Emily Chen, a leading expert in natural language processing. Michael had spent the summer working in Dr. Chen’s lab, and they had developed a novel approach to sentiment analysis that they were eager to present. \"Send me the updated dataset when you get a chance,\" Dr. Chen said, handing him her card. \"You can use emily.chen@wisc.edu.\" Michael nodded, already planning his follow-up email. Meanwhile, in the corner of the room, Jessica Miller and her friend Rachel Evans were setting up their poster on ethical AI. They were both part of the university’s ethics in technology group, and their research had been well-received in past symposiums. Jessica noticed that their poster had a small typo and whispered to Rachel, \"I’ll fix it and resend it to you. What’s your email again?\" Rachel smiled and replied, \"It’s rachel.evans@wisc.edu, and don’t worry about it—our project still looks great!\" Across the room, James Lee was chatting with a visiting professor from another university, but his mind was on the workshop he was leading later that day. He had designed an interactive session on data visualization techniques, and he hoped it would be both informative and engaging. He remembered that he still needed to send the final workshop outline to his co-presenter, Natalie Brown. He quickly typed a reminder on his phone: \"Send outline to natalie.brown@wisc.edu.\" At a nearby table, Hannah Johnson was busy reviewing her notes for a panel discussion on big data challenges. As she skimmed through her slides, she realized she needed some last-minute input from her collaborator, David Wilson. She sent him a quick message: \"Can you check slide 4 and let me know your thoughts? Send any suggestions to hannah.johnson@wisc.edu.\" By the end of the day, the symposium was deemed a success, with lively discussions, valuable networking, and promising research presented. As the crowd began to disperse, Alex gathered his things and headed toward the exit, feeling proud of what he and his peers had accomplished. Before he left, he checked his phone one last time and noticed an email from his friend and mentor, Olivia Martinez. Her email, olivia.martinez@wisc.edu, had become a frequent sight in his inbox throughout the semester. \“Great job today,” the email read. “You’re on your way to doing amazing things.”"


In [ ]:
print(email_demo)

In [ ]:
import re

# 1. Remove the formatting backslash before each @.
#    Store the result separately so the original text stays unchanged.
cleaned_text = email_demo.replace(r"\@", "@")

# 2. Describe the pattern of an email address.
pattern = r"[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}"

# 3. Find all matching addresses and store them in a Python list.
emails = re.findall(pattern, cleaned_text)

# 4. Display the number of matches and print each address.
print(f"Found {len(emails)} email addresses:\n")

for email in emails:
    print(email)

## **2. spaCy**

**14 minutes of demonstration + 6 minutes for Exercise 1.**

spaCy turns a string into a **`Doc`**, which keeps the original text and provides tokens and annotations. In this section, we inspect sentence boundaries, part-of-speech (POS) tags, lemmas, stopword flags, and named entities.

Keep capitalization and punctuation in the text passed to `nlp()`. Create a separate simplified representation afterward. The operations shown here include **lemmatization**; stemming was a separate method in Week 2.

[References: linguistic features](https://spacy.io/usage/linguistic-features) and [token attributes](https://spacy.io/api/token).

In [ ]:
# Retain the Charlie Brown example from the original notebook.
raw_name = "My name is Charlie Brown."
name_doc = nlp(raw_name)

print("First character in the string:", raw_name[0])
print("First token in the Doc:", name_doc[0].text)
print("Doc's original text:", name_doc.text)

### Sentences and tokens

The variable ending in `_text` stores the original string; the variable ending in `_doc` stores spaCy's processed document.

In [ ]:
charlie_text = (
    'Charlie Brown is a beloved character from the comic strip Peanuts, created by Charles '
    'Schulz. Known for his kind heart and persistent optimism, Charlie often finds himself '
    "facing challenges, whether it's losing at baseball or missing his chance to kick a "
    'football.'
)
charlie_doc = nlp(charlie_text)
charlie_sentences = list(charlie_doc.sents)

for sentence in charlie_sentences:
    print(sentence.text)

### Put the first sentence's annotations in a table

| Attribute | What to inspect |
|:--|:--|
| `token.text` | The token as written. |
| `token.pos_` | A readable POS label, such as `NOUN`, `PROPN` (proper noun), or `VERB`. |
| `token.lemma_` | The assigned lemma, or base form. |
| `token.is_stop` | Whether the token is flagged as a stopword. |
| `token.is_punct` | Whether the token is punctuation. |

`pd.DataFrame(...)` displays rows and named columns as a table. Here, the loop builds one row per token. You can use the table without memorizing how pandas works.

In [ ]:
annotation_rows = []
for token in charlie_sentences[0]:
    annotation_rows.append([
        token.text, token.pos_, token.lemma_, token.is_stop, token.is_punct
    ])

display(pd.DataFrame(
    annotation_rows,
    columns=["Token", "POS", "Lemma", "Stopword?", "Punctuation?"],
))

### Named Entity Recognition (NER)

`doc.ents` contains the **entity spans predicted by the model**. A span can contain several tokens, such as a person's full name. `entity.label_` gives the entity category, and `spacy.explain()` expands its abbreviation.

Inspect the text before running the cell: which names or titles would you expect it to identify? Then compare your reading with the model's output. Predictions can be missed or mislabeled, and can vary with the model version.

In [ ]:
entity_rows = []
for entity in charlie_doc.ents:
    entity_rows.append([
        entity.text, entity.label_, spacy.explain(entity.label_)
    ])

display(pd.DataFrame(entity_rows, columns=["Entity", "Label", "Explanation"]))

# Explicitly request entity highlighting; the default view is a dependency parse.
displacy.render(charlie_doc, style="ent", jupyter=True)

### A stopword flag is information; filtering is our choice

Compare a lemma-based version of **“The service was not helpful.”** with a version that also removes stopwords. We exclude punctuation and whitespace in both versions so that only the stopword decision changes.

**Predict:** which lost word would change your interpretation of this comment?

In [ ]:
review_text = "The service was not helpful."
review_doc = nlp(review_text)

all_lemmas = []
filtered_lemmas = []
for token in review_doc:
    if not token.is_punct and not token.is_space:
        lemma = token.lemma_.lower()
        all_lemmas.append(lemma)
        if not token.is_stop:
            filtered_lemmas.append(lemma)

print("Original:", review_text)
print("Lemmas, keeping stopwords:", " ".join(all_lemmas))
print("Lemmas, removing stopwords:", " ".join(filtered_lemmas))

### Count selected terms

`Counter` counts repeated items in a list. `.most_common(3)` asks for up to three items, ordered by their counts. The output pairs each item with its count.

In the exercise, the list will contain noun lemmas selected from a spaCy document.

In [ ]:
small_example = ["friend", "honey", "friend", "story"]
small_counts = Counter(small_example)

print(small_counts.most_common(3))
print("Count for 'friend':", small_counts["friend"])
print("Count for an absent term:", small_counts["library"])

### In-class Exercise 1 · spaCy: what nouns are mentioned? · 10 minutes

**Work with a partner.** This adapts the original notebook's Winnie-the-Pooh noun-counting exercise. The passage below is retained from that notebook.

1. Process the passage, then edit the **two TODO lines** to collect lowercase lemmas for tokens labeled `NOUN` or `PROPN`.
2. Run the count table and the self-check. Read the eight most frequent noun lemmas.
3. Write one observation about the result and one reason these counts need interpretation.

`NOUN` includes common nouns; `PROPN` includes proper nouns. These coarse labels cover the noun categories used by the original exercise (`NN`, `NNS`, `NNP`, `NNPS`). We are counting **tokens selected by POS**, so a multiword name can contribute several noun tokens. This is different from counting whole NER spans.

In [ ]:
pooh_text = (
    'Winnie the Pooh is a lovable, honey-obsessed bear who lives in the Hundred Acre Wood, a '
    'peaceful forest where he embarks on gentle adventures with his friends. Created by '
    'author A.A. Milne and first appearing in the 1926 book Winnie-the-Pooh, Pooh is known '
    'for his round belly, soft golden fur, and red shirt. Although often slow-witted, his '
    'simple, kind-hearted nature makes him a beloved character. He is endlessly optimistic '
    'and always willing to help his friends, including Piglet, Eeyore, Tigger, and Rabbit. '
    "Pooh's deep love for honey leads him into all sorts of small misadventures, which he "
    'approaches with a relaxed attitude and a childlike wonder. Despite his forgetfulness and '
    'lack of planning, he often stumbles upon simple wisdom, teaching readers that friendship '
    'and kindness are more important than cleverness. Pooh’s world is one of innocence, '
    'imagination, and timeless charm.'
)
pooh_doc = nlp(pooh_text)

# Inspect a few tokens before writing the selection rule.
for token in pooh_doc[:12]:
    print(token.text, "|", token.pos_, "|", token.lemma_)

In [ ]:
noun_lemmas = []
for token in pooh_doc:
    if token.pos_ in []:                    # TODO 1: add the two noun labels.
        noun_lemmas.append(token.text)      # TODO 2: use its lowercase lemma.

noun_counts = Counter(noun_lemmas)
display(pd.DataFrame(
    noun_counts.most_common(8), columns=["Noun lemma", "Count"]
))

In [ ]:
#@title Self-check: run after editing the exercise cell { display-mode: "form" }
# This checks your selection code against the model, not the model against reality.
expected_nouns = []
for token in pooh_doc:
    if token.pos_ in ["NOUN", "PROPN"]:
        expected_nouns.append(token.lemma_.lower())

if not noun_lemmas:
    print("Not yet: add the two POS labels, then rerun your exercise cell.")
elif noun_lemmas == expected_nouns:
    print("Your selection and lemma conversion match the requested rule.")
    print("Now inspect the result: are the model's labels sensible for this passage?")
else:
    print("Check that you kept only NOUN/PROPN tokens and appended lowercase lemmas.")

**Our interpretation**

A noun or name that stands out in our output: **[write here]**.

One reason these counts need interpretation: **[write here]**.

## 3. Bag-of-Words · minutes 30–40

**5 minutes of demonstration + 5 minutes for Exercise 2.**

A corpus is a collection of documents. A **document–term matrix** has one row per document and one column per vocabulary term. Each cell holds that term's count in that document.

We now use the lecture's two short documents so that we can inspect every count. Our stated policy is **lowercase surface forms, remove periods, keep stopwords, and do not lemmatize**. Thus `mining` stays `mining`, and `i` is retained. These are new example documents; we are not passing the stopword-filtered review into this section.

The next cells complete the original notebook's sequence: documents → tokens → shared vocabulary → count vectors.

In [ ]:
# Step 1: each string is one document.
documents = ["I love text mining.", "Text mining is exciting."]

# Step 2: use a simple, explicit rule for these two classroom sentences.
tokenized_documents = []
for document in documents:
    tokens = document.lower().replace(".", "").split()
    tokenized_documents.append(tokens)

print("D1:", tokenized_documents[0])
print("D2:", tokenized_documents[1])

### One shared vocabulary, in a fixed order

We list the six unique terms in the same order used in the lecture. Every row must use this order. `counts[term]` asks the `Counter` for a term's frequency and returns `0` when the term is absent.

The nested loops below walk through **each document**, then **each vocabulary term**. Each completed row becomes one document vector.

In [ ]:
# Step 3: choose a shared, ordered vocabulary.
vocabulary = ["i", "love", "text", "mining", "is", "exciting"]

# Step 4: count terms in each document, using the same column order.
bow_rows = []
for tokens in tokenized_documents:
    counts = Counter(tokens)
    row = []
    for term in vocabulary:
        row.append(counts[term])
    bow_rows.append(row)

manual_bow = pd.DataFrame(bow_rows, index=["D1", "D2"], columns=vocabulary)
display(manual_bow)

### Let `CountVectorizer` do the counting

`fit_transform()` prepares the vocabulary/feature mapping and converts the documents into a matrix. Here we supply the vocabulary explicitly to preserve the lecture's column order; in the TF-IDF example we let the vectorizer learn it.

The library's default word pattern drops single-character words. Our pattern `r"\b\w+\b"` retains `i`: `\b` is a word boundary and `\w+` is one or more word characters. For these short sentences, this produces the same tokens as the simple rule above.

The result uses **sparse storage**, which avoids storing most zero entries. `.toarray()` makes a full array for displaying this tiny example. Avoid that conversion for a large corpus.

[Reference: CountVectorizer](https://scikit-learn.org/stable/modules/generated/sklearn.feature_extraction.text.CountVectorizer.html).

In [ ]:
count_vectorizer = CountVectorizer(
    vocabulary=vocabulary,
    token_pattern=r"\b\w+\b",   # Keep one-character words, including "i".
)
count_matrix = count_vectorizer.fit_transform(documents)

library_bow = pd.DataFrame(
    count_matrix.toarray(),
    index=["D1", "D2"],
    columns=count_vectorizer.get_feature_names_out(),
)
display(library_bow)
print("Same counts as the manual table:", library_bow.equals(manual_bow))

### In-class Exercise 2 · Bag-of-Words: counts and word order · 5 minutes

**Work with a partner.** Use the vocabulary in this exact order: **`["bites", "dog", "man"]`**.

1. Read the three documents below. Replace the zeros in `predicted_counts` with your predictions, one row per document. Count repeated words separately.
2. Run the prediction cell, then run the library check to compare your table with the computed counts.
3. Explain why the first two documents receive the same or different rows, and what the third row tells you about repetition.

Use lowercase words; ignore the final period. Keep the vocabulary and document order fixed.

In [ ]:
exercise_documents = [
    "Dog bites man.",
    "Man bites dog.",
    "Dog bites dog.",
]
exercise_vocabulary = ["bites", "dog", "man"]

# TODO: replace the zeros with your predictions.
# Columns are always: bites, dog, man.
predicted_counts = [
    [0, 0, 0],   # Document 1
    [0, 0, 0],   # Document 2
    [0, 0, 0],   # Document 3
]

display(pd.DataFrame(
    predicted_counts, index=["D1", "D2", "D3"], columns=exercise_vocabulary
))

In [ ]:
# Run after making your predictions.
exercise_vectorizer = CountVectorizer(vocabulary=exercise_vocabulary)
exercise_matrix = exercise_vectorizer.fit_transform(exercise_documents)
computed_counts = exercise_matrix.toarray().tolist()

display(pd.DataFrame(
    computed_counts, index=["D1", "D2", "D3"], columns=exercise_vocabulary
))

if predicted_counts == computed_counts:
    print("Your predictions match. Now explain what the equal rows leave out.")
else:
    print("Compare each cell, check the vocabulary order, and revise your predictions.")

**Our interpretation**

The first two rows are **[the same / different]** because **[write here]**. The distinction in sentence meaning is **[write here]**.

The `dog` entry in the third row is **[write here]**, which shows **[write here]**.

## 4. TF-IDF · minutes 40–50

**Keep the documents and vocabulary visible while changing the weights.** We use the three-document cat/dog corpus from the lecture.

For the first calculation, use exactly the lecture convention:

$$TF(t,d)=\frac{\text{count of }t\text{ in }d}{\text{number of retained tokens in }d}$$
$$IDF(t)=\log_{10}\left(\frac{N}{df(t)}\right),\qquad TF\text{-}IDF(t,d)=TF(t,d)\times IDF(t)$$

Here, $N$ is the number of documents and $df(t)$ counts documents containing the term. We use no smoothing, added constant, or final vector normalization in this first calculation.

Our preprocessing policy is explicit: **lowercase, remove only `the` and `on`, and do not lemmatize.**

In [ ]:
tfidf_documents = [
    "The cat sat on the mat",
    "The dog sat on the mat",
    "The cat chased the dog",
]

# Learn the vocabulary from this corpus. Remove only these two words.
tfidf_counter = CountVectorizer(stop_words=["the", "on"])
tfidf_count_matrix = tfidf_counter.fit_transform(tfidf_documents)
tfidf_terms = tfidf_counter.get_feature_names_out()

tfidf_counts = pd.DataFrame(
    tfidf_count_matrix.toarray(),
    index=["D1", "D2", "D3"],
    columns=tfidf_terms,
)
display(tfidf_counts)

### Step A: term frequency (TF)

Add the counts **across each row** to get that document's retained-token count. Divide every entry in the row by that total. In this corpus, each document has three retained tokens.

In the code, `axis="columns"` sums across columns; `axis="index"` tells the division to align each total with its document row.

In [ ]:
document_lengths = tfidf_counts.sum(axis="columns")
classroom_tf = tfidf_counts.div(document_lengths, axis="index")

print("Retained tokens per document:", document_lengths.to_dict())
display(classroom_tf.round(3))

### Step B: document frequency (df), then IDF

`tfidf_counts > 0` asks whether each term is present in each document. Summing **down a column** counts the documents containing that term. A document contributes at most one to this count, however often the term repeats within it.

**Predict:** does `chased` occur in more documents or fewer documents than `cat`? Which should have the larger IDF?

In [ ]:
N = len(tfidf_documents)
document_frequency = (tfidf_counts > 0).sum(axis="index")
classroom_idf = np.log10(N / document_frequency)

display(pd.DataFrame({
    "Documents containing term (df)": document_frequency,
    "Classroom IDF": classroom_idf,
}).round(3))

### Step C: multiply TF by IDF

Multiply each term column by that term's IDF. The underlying numbers keep their full precision; `.round(3)` only makes the displayed table easier to read.

`cat`, `dog`, `mat`, and `sat` each occur in two documents. `chased` occurs in one. All present terms therefore tie within D1 and within D2 under this convention.

In [ ]:
classroom_tfidf = classroom_tf.mul(classroom_idf, axis="columns")
display(classroom_tfidf.round(3))

# Trace one cell back to the formula from the lecture.
chased_in_d3 = (1 / 3) * log10(3 / 1)
print("D3, 'chased':", round(chased_in_d3, 3))

### The library version: same corpus, a different numerical convention

`TfidfVectorizer` combines counting and TF-IDF weighting. Its default weights will **not** reproduce the preceding numbers because its convention differs:

| Choice | Lecture calculation above | scikit-learn defaults below |
|:--|:--|:--|
| TF before final scaling | Count ÷ retained-token count | Raw count |
| IDF | $\log_{10}(N/df)$ | $\ln((1+N)/(1+df))+1$ |
| Final row scaling | None | L2 normalization: give each row vector length 1 |

You do not need to work through the default normalization by hand today. Keep the settings documented, compare like with like, and inspect which terms receive relatively high weights. With the library's added constant, a term present in every document has IDF **1**, rather than the classroom formula's **0**.

[References: TfidfTransformer](https://scikit-learn.org/stable/modules/generated/sklearn.feature_extraction.text.TfidfTransformer.html) and [TfidfVectorizer](https://scikit-learn.org/stable/modules/generated/sklearn.feature_extraction.text.TfidfVectorizer.html).

In [ ]:
# Use exactly the same documents and stopword policy.
tfidf_vectorizer = TfidfVectorizer(stop_words=["the", "on"])
software_matrix = tfidf_vectorizer.fit_transform(tfidf_documents)
software_tfidf = pd.DataFrame(
    software_matrix.toarray(),
    index=["D1", "D2", "D3"],
    columns=tfidf_vectorizer.get_feature_names_out(),
)

print("scikit-learn defaults (different scale from the lecture table):")
display(software_tfidf.round(3))

### Closing takeaway · final minute

The count and TF-IDF tables still have **documents as rows and terms as columns**. TF-IDF changes the weights: in D3, `chased` receives a larger weight than `cat` or `dog` under both conventions because it appears in fewer documents here.

These weights describe terms relative to this corpus. A term can be central to a research question even when every document mentions it. TF-IDF does not recover word order or establish the importance of an idea to the people who produced the text.

**Connection to your project:** identify one preprocessing choice you will need to document when constructing your own corpus representation.

**End of the 50-minute demo.** The cells below are optional reference material.

## Optional references

These short examples retain selected material as reference examples.

### A. Greedy/lazy matching and the original email example

In Python's default regex mode, `.` matches a character other than a newline. `.+` is greedy: it matches as much as it can while allowing the full pattern to succeed. `.+?` tries the shortest such match.

The email pattern below reproduces the original two-address example. Treat it as a limited extraction demonstration, not a complete email validator or a guarantee that all identifiers have been removed from a text. `\w` includes word characters, `\.` matches a period, and the square brackets combine allowed characters.

In [ ]:
print("Greedy:", re.findall(r"W.+o", welcome))
print("Lazy:  ", re.findall(r"W.+?o", welcome))

contact_text = "Contact us at support@example.com or sales@example.org."
email_pattern = r"\b[\w.-]+@[\w.-]+\.\w+\b"
print("Email matches:", re.findall(email_pattern, contact_text))

### B. A shorter spelling of a familiar loop

After a loop makes sense, a list comprehension can express the same operation in one line. The following two token lists are equivalent. The timed path uses ordinary loops so that each step remains visible.

In [ ]:
tokens_from_loop = []
for token in charlie_doc:
    tokens_from_loop.append(token.text)

tokens_from_comprehension = [token.text for token in charlie_doc]
print("Same token lists:", tokens_from_loop == tokens_from_comprehension)

### C. Noun phrases and fine-grained POS labels

The original notebook also introduced noun phrases. `doc.noun_chunks` returns predicted base noun phrases; these differ from the individual noun tokens counted in Exercise 1. `.tag_` exposes a finer-grained POS label; for example, `spacy.explain("NNP")` describes that label.

In [ ]:
print("NNP means:", spacy.explain("NNP"))
print("Predicted noun phrases in the first sentence:")
for chunk in charlie_sentences[0].noun_chunks:
    print(chunk.text)

### D. Verify the library's TF-IDF formula

This optional calculation reconstructs the default weights from the count table. `np.log()` is the natural logarithm, `**2` squares entries, and `np.sqrt()` takes a square root. Each row is divided by its L2 length, $\sqrt{\sum_j w_j^2}$.

All rows in this toy corpus are nonzero. The `.replace(0, 1)` guard also makes the division safe for a hypothetical all-zero row. The check uses a tolerance because floating-point numbers can differ by tiny rounding amounts.

In [ ]:
software_idf = np.log((1 + N) / (1 + document_frequency)) + 1
unnormalized_weights = tfidf_counts.mul(software_idf, axis="columns")
row_lengths = np.sqrt((unnormalized_weights ** 2).sum(axis="columns"))
reconstructed_weights = unnormalized_weights.div(
    row_lengths.replace(0, 1), axis="index"
)

print("Reconstructed weights match scikit-learn:", np.allclose(
    reconstructed_weights.to_numpy(), software_tfidf.to_numpy()
))

## Course sources and documentation



**Software documentation for Week 3 Python Demo:** Python's [regular-expression reference](https://docs.python.org/3/library/re.html); spaCy's [model setup](https://spacy.io/usage/models), [linguistic features](https://spacy.io/usage/linguistic-features), and [Token API](https://spacy.io/api/token); scikit-learn's [CountVectorizer](https://scikit-learn.org/stable/modules/generated/sklearn.feature_extraction.text.CountVectorizer.html), [TfidfTransformer](https://scikit-learn.org/stable/modules/generated/sklearn.feature_extraction.text.TfidfTransformer.html), and [TfidfVectorizer](https://scikit-learn.org/stable/modules/generated/sklearn.feature_extraction.text.TfidfVectorizer.html).
